# CMGDB timings

Reproduce the SO(2) sphere and O(3) rows of Section 6 and Table `tab:timing`. Choose `MODE` in the first code cell. Replay reads the saved JSON records in `data/timings/`; before the final measurements exist, it shows the available short validation records separately. Train runs every published row, takes about 40 minutes on the reference machine, and refuses to start unless all three load averages are below 2 and at least 16 GB of physical memory is free. Set `DIRECTORY` to a new folder to preserve an earlier run.

From a terminal at the repository root, use `python scripts/run_timings.py all` for the final run. For short validation only, use `python scripts/run_timings.py smoke`. Run `python scripts/run_timings.py --help` for individual rows and repeat settings.

In [1]:
MODE = "replay"  # "replay": saved records; "train": recompute all six rows
DIRECTORY = "data/timings"

from morsegraphs import timing
run = timing.notebook_run(MODE, DIRECTORY)
run

## Machine

Every JSON record includes its machine, package versions, native-library checksum, numerical thread counts, free memory, load averages, individual stage measurements, and graph. Each row runs in a fresh process with `OMP_NUM_THREADS`, `OPENBLAS_NUM_THREADS`, `VECLIB_MAXIMUM_THREADS`, and `MKL_NUM_THREADS` set to one. The original experiment used an Apple M4 Pro with 14 cores and 48 GiB of RAM. Timings depend on the machine and its load.

In [2]:
timing.notebook_machines(run)

CPU,System,CPUs,RAM,Python,NumPy,CMGDB,Numerical threads
None,Darwin 27.2.0,14,unknown,3.13.18,2.3.5,1.5.3+fork.7,1 each


## Section 6

These are the four total times quoted in the discussion. Validation records check the implementation on a shared machine; they do not replace final timing measurements. Small original-space grids never stand in for the published rows.

In [3]:
timing.notebook_discussion(run)

Computation,Section 6,Measured,Status
sphere-quotient,3.1 ms,2.6 ms,validation
o3-quotient,1.8 ms,1.7 ms,validation
sphere-uniform,47 s,pending,waiting for idle run
o3-uniform,1.01e+03 s,pending,waiting for idle run


## Stage accounting and qualifications

Map construction includes image hulls and their covers. On uniform grids, the cover is measured separately with the native lazy map graph; its time is removed from the Morse-graph time. On adaptive grids the cover remains in the Morse-graph column. Conley time subtracts the repeated graph phase and callback time from the native Conley call. On the O(3) quotient, the table uses the within-call per-set split because the difference of separate calls is near the timer floor. Raw negative differences and undefined native indices remain in the JSON records. The table displays values below 1 ms as `<1 ms` and totals as sums of stage medians. Sphere rows use five timed repeats after one warmup; O(3) original-space rows use one repeat without a warmup, and its quotient uses five repeats after one warmup with 116 inner iterations.

All sphere rows use `BoxMapData` from samples of the time-ten map with rotation removed; empty cubes map outside the box. The quotient has $2^7$ cells and the original uniform grid has $2^{21}$. The O(3) map is the time-$\log 3$ flow with floating-point coordinate hulls and relative outward padding $10^{-12}$; the quotient has $2^5\times2^5$ cells and the original uniform grid has $2^{23}$. The original-space cell counts cover the bounding boxes, not only the physical state spaces. Native indices belong to these computed multivalued maps. For the published sphere data map, the circle indices are undefined and the boundary poles have zero index; the published original-space O(3) graph has 164 components, including 160 with zero index. These numerical outputs do not certify physical index transfer.

The table pairs each manuscript row with the corresponding new record. Adaptive cell counts on measured rows come from the final native grid; pending rows retain the manuscript count.

In [4]:
timing.notebook_table(run)

Dimension,G,Grid,Cells,Source,Map,Morse graph,Conley index,Total
3,SO(2),uniform,2^21,manuscript,31 s,1.1 s,14 s,47 s
3,SO(2),uniform,2^21,waiting for idle run,pending,pending,pending,pending
3,SO(2),adaptive,"17,464",manuscript,2.9 s,60 ms,21 s,24 s
3,SO(2),adaptive,"17,464",waiting for idle run,pending,pending,pending,pending
1,SO(2),uniform,2^7,manuscript,2.3 ms,<1 ms,<1 ms,3.1 ms
1,SO(2),uniform,2^7,validation,1.9 ms,<1 ms,<1 ms,2.6 ms
4,O(3),uniform,2^23,manuscript,16 s,4.9 s,991 s,1.01e+03 s
4,O(3),uniform,2^23,waiting for idle run,pending,pending,pending,pending
4,O(3),adaptive,"109,536",manuscript,2.3 s,5.8e+02 ms,943 s,946 s
4,O(3),adaptive,"109,536",waiting for idle run,pending,pending,pending,pending
